# The Shooting Method
&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/differential_equations/boundary_value_problems/shooting_method.ipynb)

The **shooting method** turns a two-point Boundary Value Problem on $[a, b]$ into a sequence of Initial Value Problems (IVPs), just like aiming a cannon toward a distant target:

1. **Guess** the missing initial condition(s) at $a$ (e.g., an initial slope $y'(a) = s$).
2. **Integrate** the ODE from $a$ to $b$ using an IVP solver (`solve_ivp`).
3. **Evaluate the residual** by comparing the computed value at $b$ with the required boundary value.
4. **Refine** the guess with a root-finding algorithm (`root`) until the trajectory hits the target at $b$.

In [ ]:
try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Pros and Cons

### Advantages
- **Intuitive and direct**: Reuses accurate adaptive IVP solvers without assembling large global matrices.
- **Good for linear problems**: Particularly useful for linear problems that can be refined efficiently.
- **Handles nonlinearity naturally**: Nonlinear ODEs require only a low-dimensional root-finding loop over the missing initial values.
- **Reduced complexity in low dimensions**: For low-dimensional problems, it often requires fewer computations.  

### Drawbacks
- **Sensitivity and instability**: If the ODE has exponentially growing modes, tiny changes in the initial guess can blow up at the far boundary.
- **Guess dependence**: A poor initial guess may cause the root finder to diverge or miss alternative solutions.
- **Dimension limit**: Restricted to 1D domains; it does not generalize directly to multi-dimensional PDEs.
- **Potential for numerical instability**: Error can accumulate over the integration intervals, especially for stiff ODEs. 

## Example: Ballistics

Consider launching a projectile vertically under gravity $g = 9.8\,\text{m/s}^2$, starting from the ground $y(0) = 0\,\text{m}$ and required to be at altitude $y(5) = 50\,\text{m}$ after $5\,\text{s}$:

$$\frac{\mathrm{d}^2 y}{\mathrm{d}t^2} = -g, \qquad y(0) = 0, \qquad y(5) = 50$$

To apply an IVP solver, rewrite the second-order ODE as a first-order system for altitude $y$ and velocity $v$:

$$\begin{aligned}
\frac{\mathrm{d}y}{\mathrm{d}t} &= v \\
\frac{\mathrm{d}v}{\mathrm{d}t} &= -g
\end{aligned}$$

We shoot with candidate launch velocities $v_0 = v(0)$ until $y(5) - 50 = 0$.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def shooting_ballistics
:end-before: def rbf_poisson_2d
```

In [ ]:
exe.display_source(exe.shooting_ballistics)

t, y, v, v0_star = exe.shooting_ballistics()
print(f"Initial velocity v0: {v0_star:.2f} m/s")
print(f"Altitude at t=5 s:   {y[-1]:.2f} m")

exe.plot_shooting_ballistics().show()